In [ ]:
"""" This is to check if combining the particular project's info with all other 46 projects' info improves the AUC 
Instead of seeing if we can predict a project with no history, we are checking whether, if a project has a little history, does adding other projects'
data help beyond its own? """

In [ ]:
# Getting the data from the other notebook 

In [2]:
import glob, os, numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

DATA_DIR = r"C:\Users\ramku\OneDrive\Documents\Research Practicum\Execution\Reopened bugs dataset"

NUM = ['changelog_Weekday','changelog_Month_day','changelog_Month','changelog_Day_of_year',
       'Number_of_comments','changelog_Priority_changed','Time_days','Number_of_files_changed',
       'updated_description_size','accumulated_comment_size','reporter_experience','assignee_experience']

files = sorted(glob.glob(os.path.join(DATA_DIR, "joined_project_dataframe_*.csv")))
num_list, emb_list, y_list, proj_list = [], [], [], []
for f in files:
    name = os.path.basename(f).replace("joined_project_dataframe_", "").replace(".csv", "")
    df  = pd.read_csv(f, low_memory=False)
    emb = np.load(os.path.join(DATA_DIR, f"emb_{name}.npy"))
    assert len(df) == len(emb), f"{name}: rows {len(df)} vs emb {len(emb)}"   # safety check
    num_list.append(df[NUM].fillna(0).astype(float).values)
    emb_list.append(emb)
    y_list.append(df["Reopened"].values)
    proj_list.append(np.array([name] * len(df)))

X_num_all = np.vstack(num_list)                               # numeric features, all bugs
X_emb_all = np.vstack(emb_list)                               # embeddings, all bugs
y_all     = np.concatenate(y_list)                           # labels, all bugs
proj_all  = np.concatenate(proj_list)                        # which project each bug is from
X_all     = np.hstack([X_num_all, X_emb_all]).astype(np.float32)   # numeric + embeddings combined

print("Numeric:", X_num_all.shape, "| Embeddings:", X_emb_all.shape,
      "| Labels:", y_all.shape, "reopened:", int(y_all.sum()), "| Projects:", len(np.unique(proj_all)))

Numeric: (168881, 12) | Embeddings: (168881, 384) | Labels: (168881,) reopened: 8886 | Projects: 47


In [ ]:
# First we will just try it on one project alone - ARIES

In [2]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_all = np.hstack([X_num_all, X_emb_all]).astype(np.float32)   # numeric + embeddings

P = "ARIES"
tgt    = np.where(proj_all == P)[0]      # this project's bugs
others = np.where(proj_all != P)[0]      # all the other projects' bugs
yP = y_all[tgt]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
within_aucs, aug_aucs = [], []
for tr_local, te_local in skf.split(tgt, yP):
    tr_idx, te_idx = tgt[tr_local], tgt[te_local]       # same test fold for both models

    # (1) within-project only
    rf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[tr_idx], y_all[tr_idx])
    within_aucs.append(roc_auc_score(y_all[te_idx], rf_w.predict_proba(X_all[te_idx])[:, 1]))

    # (2) augmented = own training fold + all other projects
    aug_tr = np.concatenate([tr_idx, others])
    rf_a = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[aug_tr], y_all[aug_tr])
    aug_aucs.append(roc_auc_score(y_all[te_idx], rf_a.predict_proba(X_all[te_idx])[:, 1]))

print(f"{P}: within-only = {np.mean(within_aucs):.3f}   augmented (+others) = {np.mean(aug_aucs):.3f}")

ARIES: within-only = 0.597   augmented (+others) = 0.633


In [3]:
# Trying it on a small project now (OODT as it has 17 reopened bugs)
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

P = "OODT"          
tgt    = np.where(proj_all == P)[0]
others = np.where(proj_all != P)[0]
yP = y_all[tgt]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
within_aucs, aug_aucs = [], []
for tr_local, te_local in skf.split(tgt, yP):
    tr_idx, te_idx = tgt[tr_local], tgt[te_local]
    rf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[tr_idx], y_all[tr_idx])
    within_aucs.append(roc_auc_score(y_all[te_idx], rf_w.predict_proba(X_all[te_idx])[:, 1]))
    aug_tr = np.concatenate([tr_idx, others])
    rf_a = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[aug_tr], y_all[aug_tr])
    aug_aucs.append(roc_auc_score(y_all[te_idx], rf_a.predict_proba(X_all[te_idx])[:, 1]))

print(f"{P} (reopened {int(yP.sum())}): within-only = {np.mean(within_aucs):.3f}   augmented = {np.mean(aug_aucs):.3f}   delta = {np.mean(aug_aucs)-np.mean(within_aucs):+.3f}")

OODT (reopened 17): within-only = 0.538   augmented = 0.594   delta = +0.056


In [4]:
# Trying it on a small project now (PARQUET as it has 17 reopened bugs)
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

P = "PARQUET"          
tgt    = np.where(proj_all == P)[0]
others = np.where(proj_all != P)[0]
yP = y_all[tgt]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
within_aucs, aug_aucs = [], []
for tr_local, te_local in skf.split(tgt, yP):
    tr_idx, te_idx = tgt[tr_local], tgt[te_local]
    rf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[tr_idx], y_all[tr_idx])
    within_aucs.append(roc_auc_score(y_all[te_idx], rf_w.predict_proba(X_all[te_idx])[:, 1]))
    aug_tr = np.concatenate([tr_idx, others])
    rf_a = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[aug_tr], y_all[aug_tr])
    aug_aucs.append(roc_auc_score(y_all[te_idx], rf_a.predict_proba(X_all[te_idx])[:, 1]))

print(f"{P} (reopened {int(yP.sum())}): within-only = {np.mean(within_aucs):.3f}   augmented = {np.mean(aug_aucs):.3f}   delta = {np.mean(aug_aucs)-np.mean(within_aucs):+.3f}")

PARQUET (reopened 16): within-only = 0.678   augmented = 0.666   delta = -0.012


In [5]:
# SENTRY - 22 reopened bugs
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

P = "SENTRY"          
tgt    = np.where(proj_all == P)[0]
others = np.where(proj_all != P)[0]
yP = y_all[tgt]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
within_aucs, aug_aucs = [], []
for tr_local, te_local in skf.split(tgt, yP):
    tr_idx, te_idx = tgt[tr_local], tgt[te_local]
    rf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[tr_idx], y_all[tr_idx])
    within_aucs.append(roc_auc_score(y_all[te_idx], rf_w.predict_proba(X_all[te_idx])[:, 1]))
    aug_tr = np.concatenate([tr_idx, others])
    rf_a = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                  class_weight='balanced').fit(X_all[aug_tr], y_all[aug_tr])
    aug_aucs.append(roc_auc_score(y_all[te_idx], rf_a.predict_proba(X_all[te_idx])[:, 1]))

print(f"{P} (reopened {int(yP.sum())}): within-only = {np.mean(within_aucs):.3f}   augmented = {np.mean(aug_aucs):.3f}   delta = {np.mean(aug_aucs)-np.mean(within_aucs):+.3f}")

SENTRY (reopened 22): within-only = 0.470   augmented = 0.397   delta = -0.073


In [ ]:
# Testing it out on projects having the top-least 10 reopened bugs
import numpy as np, pandas as pd, os
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

out_csv = os.path.join(DATA_DIR, "small_projects_full.csv")

# the 3 already run as pilots (same method) — pre-filled so we don't redo them
seed = [
    {"project": "OODT",    "n_reopened": 17, "within_only": 0.538, "augmented": 0.594, "delta": 0.056},
    {"project": "PARQUET", "n_reopened": 16, "within_only": 0.678, "augmented": 0.666, "delta": -0.012},
    {"project": "SENTRY",  "n_reopened": 22, "within_only": 0.470, "augmented": 0.397, "delta": -0.073},
]
rows = pd.read_csv(out_csv).to_dict("records") if os.path.exists(out_csv) else []
have = set(r["project"] for r in rows)
for s in seed:
    if s["project"] not in have:
        rows.append(s); have.add(s["project"])
pd.DataFrame(rows).to_csv(out_csv, index=False)
done = have

small10 = sorted([p for p in np.unique(proj_all) if (y_all[proj_all == p] == 1).sum() >= 10],
                 key=lambda p: int((y_all[proj_all == p] == 1).sum()))[:10]

for i, P in enumerate(small10, 1):
    if P in done:
        print(f"[{i}/10] {P}: already have it, skip"); continue
    tgt = np.where(proj_all == P)[0]; others = np.where(proj_all != P)[0]
    yP = y_all[tgt]
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    w_aucs, a_aucs = [], []
    for tr_local, te_local in skf.split(tgt, yP):
        tr_idx, te_idx = tgt[tr_local], tgt[te_local]
        rf_w = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                      class_weight='balanced').fit(X_all[tr_idx], y_all[tr_idx])
        w_aucs.append(roc_auc_score(y_all[te_idx], rf_w.predict_proba(X_all[te_idx])[:, 1]))
        aug_tr = np.concatenate([tr_idx, others])
        rf_a = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=4,
                                      class_weight='balanced').fit(X_all[aug_tr], y_all[aug_tr])
        a_aucs.append(roc_auc_score(y_all[te_idx], rf_a.predict_proba(X_all[te_idx])[:, 1]))
    w, a = np.mean(w_aucs), np.mean(a_aucs)
    rows.append({"project": P, "n_reopened": int(yP.sum()), "within_only": round(w, 3),
                 "augmented": round(a, 3), "delta": round(a - w, 3)})
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print(f"[{i}/10] {P}: within={w:.3f}  augmented={a:.3f}  delta={a-w:+.3f}  (reopened {int(yP.sum())})")

m = pd.DataFrame(rows)
print("\n=== SMALL-10 SUMMARY ===")
print("Augmented > within in", int((m['delta'] > 0).sum()), "/", len(m), "| Mean delta:", round(m['delta'].mean(), 3))

[1/10] ORC: within=0.335  augmented=0.422  delta=+0.087  (reopened 12)
[2/10] PARQUET: already have it, skip
[3/10] OODT: already have it, skip
[4/10] SENTRY: already have it, skip
[5/10] GERONIMO: within=0.652  augmented=0.492  delta=-0.160  (reopened 46)
[6/10] ASTERIXDB: within=0.681  augmented=0.586  delta=-0.095  (reopened 47)
[7/10] BIGTOP: within=0.607  augmented=0.535  delta=-0.071  (reopened 47)
[8/10] FINERACT: within=0.689  augmented=0.538  delta=-0.151  (reopened 50)
[9/10] JAMES: within=0.552  augmented=0.611  delta=+0.060  (reopened 52)
